# Beta Pictoris orbit fit

In [ ]:
import jax
import numpyro

# For multi-core parallelism (useful when running multiple MCMC chains in parallel)
numpyro.set_host_device_count(4)

# For CPU (use "gpu" for GPU)
numpyro.set_platform("cpu")

jax.config.update("jax_enable_x64", True)

## Relative astrometry data

In [ ]:
import pandas as pd
# TODO: Download from somewhere online to make reprod easier
df_astro = pd.read_csv("../data/astrometry.csv")

In [ ]:
import numpy as np

t_astro = df_astro.time.values
sep = df_astro.sep.values
pa = df_astro.pa.values
sep_err = df_astro.sep_err.values
pa_err = df_astro.pa_err.values

pa_rad = np.radians(pa)
ra = sep * np.sin(pa_rad)
dec = sep * np.cos(pa_rad)
pa_err_rad = np.radians(pa_err)
dec_err = np.sqrt(
    (np.cos(pa_rad)**2 * sep_err**2) +
    (sep**2 * np.sin(pa_rad)**2 * pa_err_rad**2)
)
ra_err = np.sqrt(
    (np.sin(pa_rad)**2 * sep_err**2) +
    (sep**2 * np.cos(pa_rad)**2 * pa_err_rad**2)
)
df_astro["ra"] = ra
df_astro["ra_err"] = ra_err
df_astro["dec"] = dec
df_astro["dec_err"] = dec_err

In [ ]:
import matplotlib.pyplot as plt

_, ax = plt.subplots(figsize=(4, 4))
ax.errorbar(df_astro.ra, df_astro.dec, xerr=df_astro.ra_err, yerr=df_astro.dec_err, fmt="k.")
ax.plot(0, 0, "k*", markersize=15)
ax.axis("equal")
ax.invert_xaxis()
ax.set_xlabel("$\\Delta$ RA [mas]")
ax.set_ylabel("$\\Delta$ Dec [mas]")
plt.tight_layout()
plt.show()

In [ ]:
fix, axs = plt.subplots(nrows=2, sharex=True, figsize=(6, 8))
axs[0].errorbar(df_astro.time, df_astro.sep, yerr=df_astro.sep_err, fmt="k.")
axs[0].set_ylabel("Sep [mas]")
axs[1].errorbar(df_astro.time, df_astro.pa, yerr=df_astro.pa_err, fmt="k.")
axs[1].set_ylabel("PA [deg]")

axs[-1].set_xlabel("Time [MJD]")
plt.show()

## RV Data

In [ ]:
df_rv = pd.read_csv("../data/rv.csv")

t_rv = df_rv.mjd.values
rv = df_rv.rv.values
rv_err = df_rv.rv_err.values

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(df_rv.mjd, df_rv.rv, yerr=df_rv.rv_err, fmt="k.")
ax.set_ylabel("RV [m/s]")
ax.set_xlabel("MJD")
plt.show()

## Forward model

### Astrometric model

In [ ]:
from jaxoplanet.orbits.keplerian import System, Body, Central
import jax.numpy as jnp

def build_system(params):
    system = System(
        Central(mass=params["m_tot"], radius=1.0)
    ).add_body(
        Body(
            time_transit=params["Tc"],
            period=params.get("P", None),
            semimajor=params.get("a", None),
            inclination=params["i"],
            eccentricity=params["e"],
            omega_peri=params["omega"],
            sin_asc_node=jnp.sin(params["Omega"]),
            cos_asc_node=jnp.cos(params["Omega"]),
            parallax=params["parallax"],
            mass=params.get("mass", None)
        )
    )
    return system

def astrometry_model(time, params):
    system = build_system(params)
    sep, pa = system.bodies[0].relative_angles(time, parallax=params["parallax"])
    sep = sep * 1e3
    pa = pa / deg
    return sep, pa % 360

In [ ]:
import astropy.units as u

deg = np.pi / 180
yr = 365.25
mjup = (1 * u.Mjup).to(u.Msun).value

init_params = {
    "P": 20.3 * yr,
    "Tc": 58010,
    "i": 89 * deg,
    "e": 0.0,
    "omega": 13 * deg,
    "Omega": 32 * deg,
    "parallax": 51.44 * 1e-3,
    "m_tot": 1.4,
    "mass": 10.0 * mjup,
}

In [ ]:
t_fine = np.linspace(t_astro.min() - 10 * yr, t_astro.max() + 10 * yr, num=1000)
sep_fine, pa_fine = astrometry_model(t_fine, init_params)
sep_mod, pa_mod = astrometry_model(t_astro, init_params)

fix, axs = plt.subplots(nrows=4, sharex=True, figsize=(6, 8), gridspec_kw={"height_ratios": (2, 1, 2, 1)})
axs[0].errorbar(df_astro.time, df_astro.sep, yerr=df_astro.sep_err, fmt="k.")
axs[0].plot(t_fine, sep_fine)
axs[0].set_ylabel("Sep [mas]")

axs[1].errorbar(df_astro.time, df_astro.sep - sep_mod, yerr=df_astro.sep_err, fmt="k.")
axs[1].set_ylabel("Sep residuals [mas]")
axs[1].axhline(0, linestyle="--")

axs[2].errorbar(df_astro.time, df_astro.pa, yerr=df_astro.pa_err, fmt="k.")
axs[2].plot(t_fine, pa_fine)
axs[2].set_ylabel("PA [deg]")

axs[3].errorbar(df_astro.time, df_astro.pa - pa_mod, yerr=df_astro.sep_err, fmt="k.")
axs[3].axhline(0, linestyle="--")
axs[3].set_ylabel("PA residuals [deg]")

axs[-1].set_xlabel("Time [MJD]")
plt.show()

In [ ]:
ra_fine = sep_fine * np.sin(pa_fine * deg)
dec_fine = sep_fine * np.cos(pa_fine * deg)
_, ax = plt.subplots(figsize=(4, 4))
ax.errorbar(df_astro.ra, df_astro.dec, xerr=df_astro.ra_err, yerr=df_astro.dec_err, fmt="k.")
ax.plot(ra_fine, dec_fine)
ax.plot(0, 0, "k*", markersize=15)
ax.axis("equal")
ax.invert_xaxis()
ax.set_xlabel("$\\Delta$ RA [mas]")
ax.set_ylabel("$\\Delta$ Dec [mas]")
plt.tight_layout()
plt.show()

### RV Model

In [ ]:
def rv_model(time, params):
    system = build_system(params)
    return system.radial_velocity(time)[0]

In [ ]:
t_fine_rv = np.linspace(t_rv.min(), t_rv.max(), num=1000)
rv_fine = rv_model(t_fine_rv, init_params)
rv_mod = rv_model(t_rv, init_params)

fig, axs = plt.subplots(2, 1, figsize=(8, 8), gridspec_kw={"height_ratios": (2, 1)})
axs[0].errorbar(df_rv.mjd, df_rv.rv, yerr=df_rv.rv_err, fmt="k.")
axs[0].plot(t_fine_rv, rv_fine)
axs[0].set_ylabel("RV [m/s]")

axs[1].errorbar(df_rv.mjd, df_rv.rv - rv_mod, yerr=df_rv.rv_err, fmt="k.")
axs[1].axhline(0, linestyle="--")
axs[1].set_ylabel("RV residuals [m/s]")

axs[-1].set_xlabel("MJD")

plt.show()

## Probabilistic model

In [ ]:
import numpyro
from jaxoplanet import constants
from numpyro import distributions as dist, infer
from numpyro_ext import distributions as distx

def model():
    tc = numpyro.sample("tc", dist.Uniform(57000, 59000))
    hk = numpyro.sample("hk", distx.UnitDisk())
    h, k = hk[..., 0], hk[..., 1]
    numpyro.deterministic("h", h)
    numpyro.deterministic("k", k)
    e = numpyro.deterministic("e", h**2 + k**2)
    omega = numpyro.deterministic("omega", jnp.arctan2(k, h))

    loga = numpyro.sample("loga", dist.Uniform(jnp.log(1), jnp.log(100)))
    a = numpyro.deterministic("a", jnp.exp(loga))

    mass = numpyro.sample("mass", dist.Uniform(1, 20))

    Omega = numpyro.sample("Omega", dist.Uniform(18 * deg, 90 * deg))

    sini = numpyro.sample("sini", dist.Uniform(-1, 1))
    i = numpyro.deterministic("i", jnp.arcsin(sini))

    plx = numpyro.sample("plx", dist.Normal(51.44e-3, 0.12e-3))

    gamma = numpyro.sample("gamma", dist.Uniform(-100, 100))

    m_tot = numpyro.sample("m_tot", dist.Uniform(1.4, 2.0))

    params = {
        "m_tot": m_tot,
        "mass": mass * constants.M_jup,
        "Tc": tc,
        # "P": P,
        "a": a * constants.au,
        "e": e,
        "i": i,
        "omega": omega,
        "Omega": Omega,
        "parallax": plx,
    }

    rv_mod = rv_model(t_rv, params) - gamma
    sep_mod, pa_mod = astrometry_model(t_astro, params)
    sep_tot_err = sep_err
    pa_tot_err = pa_err
    rv_tot_err = rv_err

    # Likelihood on sep and pa
    # The likelihood for pa accounts for wrapping of the angle
    numpyro.sample("sep_obs", dist.Normal(loc=sep_mod, scale=sep_tot_err), obs=sep)
    numpyro.sample("rv_obs", dist.Normal(loc=rv_mod, scale=rv_tot_err), obs=rv)
    pa_diff = jnp.arctan2(
        jnp.sin(pa_mod * deg - pa * deg), jnp.cos(pa_mod * deg - pa * deg)
    )
    numpyro.sample("pa_obs", dist.Normal(loc=pa_diff, scale=pa_tot_err * deg), obs=0.0)

    # Save finer grid for plotting
    sep_dense, pa_dense = astrometry_model(t_fine, params)
    rv_dense = rv_model(t_fine_rv, params) - gamma
    sep_save = numpyro.deterministic("sep_save", sep_dense)
    pa_save = numpyro.deterministic("pa_save", pa_dense)
    rv_save = numpyro.deterministic("rv_save", rv_dense)


In [ ]:
import arviz as az
import corner
n_prior_samples = 2000

key = jax.random.key(0)

key, subkey = jax.random.split(key)
prior_samples = infer.Predictive(model, num_samples=n_prior_samples)(subkey)

converted_prior_samples = {
    f"{p}": np.expand_dims(prior_samples[p], axis=0) for p in prior_samples
}
var_names = ["tc", "h", "k", "loga", "mass", "Omega", "sini", "plx", "gamma", "m_tot"]
det_names = ["e", "omega", "a", "i"]
prior_idata = az.from_dict(converted_prior_samples)

# corner.corner(prior_idata, var_names=var_names, group="prior")
corner.corner(prior_idata, var_names=var_names + det_names)
plt.show()

In [ ]:
rng = np.random.default_rng()

n_plots = 100
plot_inds = rng.integers(n_prior_samples, size=n_plots)

fig, axs = plt.subplots(nrows=2, sharex=True, figsize=(6, 8))
axs[0].errorbar(df_astro.time, df_astro.sep, yerr=df_astro.sep_err, fmt="k.")
axs[0].set_ylabel("Sep [mas]")
axs[1].errorbar(df_astro.time, df_astro.pa, yerr=df_astro.pa_err, fmt="k.")
axs[1].set_ylabel("PA [deg]")
for idx in plot_inds:
    axs[0].plot(t_fine, prior_idata.posterior["sep_save"].values[0, idx], "C1", alpha=0.1)
    axs[1].plot(t_fine, prior_idata.posterior["pa_save"].values[0, idx], "C1", alpha=0.1)

axs[-1].set_xlabel("Time [MJD]")
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(df_rv.mjd, df_rv.rv, yerr=df_rv.rv_err, fmt="k.")
for idx in plot_inds:
    ax.plot(t_fine, prior_idata.posterior["rv_save"].values[0, idx], "C1", alpha=0.1)
ax.set_ylabel("RV [m/s]")
ax.set_xlabel("MJD")
plt.show()

## Optimization


In [ ]:
import numpyro_ext.optim as optimx

opt_sites = [
    ["tc"],
    ["hk", "Omega"],
    ["logP", "loga", "tc"],
    ["mass", "sini", "plx", "gamma"],
    None,
]
start = init_params
start["mass"] *= constants.M_jup
start["loga"] = np.log(9.8)
start["hk"] = np.array([0, 0])
start["sini"] = np.sin(start["i"])
for i, sites in enumerate(opt_sites):
    print(f"Optimizing {sites}")
    if i == 0:
        #init = infer.init_to_value(values=start)
        init = infer.init_to_median()
    else:
        init = infer.init_to_value(values=map_soln)
    key, subkey = jax.random.split(key)
    map_soln = optimx.optimize(model, sites=sites, init_strategy=init)(subkey)

In [ ]:
rng = np.random.default_rng()

n_plots = 100
plot_inds = rng.integers(n_prior_samples, size=n_plots)

fig, axs = plt.subplots(nrows=2, sharex=True, figsize=(6, 8))
axs[0].errorbar(df_astro.time, df_astro.sep, yerr=df_astro.sep_err, fmt="k.")
axs[0].set_ylabel("Sep [mas]")
axs[1].errorbar(df_astro.time, df_astro.pa, yerr=df_astro.pa_err, fmt="k.")
axs[1].set_ylabel("PA [deg]")
axs[0].plot(t_fine, map_soln["sep_save"], "C1")
axs[1].plot(t_fine, map_soln["pa_save"], "C1")

axs[-1].set_xlabel("Time [MJD]")
plt.show()

In [ ]:
sep_fine = map_soln["sep_save"]
pa_fine = map_soln["pa_save"]
ra_fine = sep_fine * np.sin(pa_fine * deg)
dec_fine = sep_fine * np.cos(pa_fine * deg)
_, ax = plt.subplots(figsize=(4, 4))
ax.errorbar(df_astro.ra, df_astro.dec, xerr=df_astro.ra_err, yerr=df_astro.dec_err, fmt="k.")
ax.plot(ra_fine, dec_fine)
ax.plot(0, 0, "k*", markersize=15)
ax.axis("equal")
ax.invert_xaxis()
ax.set_xlabel("$\\Delta$ RA [mas]")
ax.set_ylabel("$\\Delta$ Dec [mas]")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(df_rv.mjd, df_rv.rv, yerr=df_rv.rv_err, fmt="k.")
ax.plot(t_fine, map_soln["rv_save"], "C1", alpha=0.1)
ax.set_ylabel("RV [m/s]")
ax.set_xlabel("MJD")
plt.show()